# Tools

This notebook contains all the tools that will be used by the agent.

In [ ]:
import requests
import sqlite3
import pandas as pd

from smolagents import tool, Tool

#from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun

## Creating simple tool

A Python function must be annotated with `@tool`. It should also have a docstring describing what does the function do, what does it return and the description of its parameters.

### City to location

The following function look up at latitude and longitude of a city.

In [ ]:
# TODO: Load CSV file containing latitude, longitude and altitude of cities
# https://github.com/bahar/WorldCityLocations/tree/master
df = pd.read_csv('/content/data/cities_latlng.csv', sep=';')
print(df.head())


   id      country            city   latitude  longitude  altitude
0   1  Afghanistan           Kabul  34.516667  69.183334      1808
1   2  Afghanistan        Kandahar  31.610000  65.699997      1015
2   3  Afghanistan  Mazar-e Sharif  36.706944  67.112221       369
3   4  Afghanistan           Herat  34.340000  62.189999       927
4   5  Afghanistan       Jalalabad  34.420000  70.449997       573


In [ ]:
# TODO: Explore the loaded dataframe


In [ ]:
# TODO: Add tool description

@tool
def get_latlng(city: str) -> dict:
  """
  Return the latitude, longitude and altitude of a city in a dictionary

  Args:
    city: The name of the city

  Returns:
    any: A dictionary containing the latitude, longitude and altitude of the city

  Example:
    get_latlng('Singapore')
  """
  r = df.query(f"city.str.lower() == '{city.lower()}'")
  return { 'city': city, 'latitude': r.iloc[0]['latitude'], 'longitude': r.iloc[0]['longitude'], 'altitude': r.iloc[0]['altitude'] }

In [ ]:
# TODO: Test get_latlng method
# case insensitive search
get_latlng('singapore')

{'city': 'singapore',
 'latitude': np.float64(1.29027),
 'longitude': np.float64(103.851959),
 'altitude': np.int64(164)}

### Temperature at latitude and longitude

The following function lookup the weather at the given latitude and longtude.

In [ ]:
# TODO: Add tool description

@tool
def get_temperature(latitude: float, longitude: float) -> dict:
  """
  Return the temperature of a city given by its latitude and longitude

  Args:
    latitude: latitude of the city
    longitude: longitude of the city

  Returns:
    any: A dictionary containing the temperature of the city and the temperature unit

  Example:
    get_temperature(1.2902, 103.851959)
  """
  url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
  resp = requests.get(url)
  j = resp.json()
  if resp.status_code >= 400:
     raise Exception(j['reason'])
  temperature = j['current']['temperature_2m']
  units = j['current_units']['temperature_2m']
  return { "temperature_unit": units, "temperature": temperature }

In [ ]:
# TODO: Test get_temperature method
get_temperature(1.2902, 103.851959)

{'temperature_unit': '°C', 'temperature': 29.6}

### Query relational database

The following function queries a relational database (SQLite) view called `album_track`. The table's schema is as follows:
| Field name  | Type          |
|-------------|---------------|
| AlbumId     | integer       |
| Title       | nvarchar(160) |
| track_name  | nvarchar(200) |
| artist_name | nvarchar(120) |
| duration    | integer       |
| composer    | nvarchar(220) |


In [ ]:
# TODO: Add tool description

@tool
def query_album_track(query: str) -> list:
  """
  Perform SQL query on the album_track table. Returns the result as an array of tuples.
  The table name is album_track and the schema is as follows:
    AlbumId: integer
    Title: nvarchar(160)
    track_name: nvarchar(200)
    artist_name: nvarchar(120)
    duration: integer
    composer: nvarchar(220)
  The duration is in milliseconds

  Args:
    query: A valid SQL query in SQLite dialect

  Returns:
    list: An array of tuples containing the result of the query

  Example:
    query_album_track("select * from album_track order by track_name limit 10")
  """

  database = "data/chinook_sqlite.sqlite"
  conn = sqlite3.connect(database)
  try:
     cursor = conn.cursor()
     rows = cursor.execute(query)
     return rows.fetchall()
  finally:
     conn.close()

In [ ]:
# TODO: Test the query_album_track function
query_album_track("select * from album_track order by track_name limit 10")

[(239, 'War', '"40"', 'U2', 157962, 'U2'),
 (231, 'Lost, Season 2', '"?"', 'Lost', 2782333, None),
 (281,
  'Sir Neville Marriner: A Celebration',
  '"Eine Kleine Nachtmusik" Serenade In G, K. 525: I. Allegro',
  'Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner',
  348971,
  'Wolfgang Amadeus Mozart'),
 (11,
  'Out Of Exile',
  '#1 Zero',
  'Audioslave',
  299102,
  'Cornell, Commerford, Morello, Wilk'),
 (255,
  'Instant Karma: The Amnesty International Campaign to Save Darfur',
  '#9 Dream',
  'U2',
  278312,
  None),
 (48,
  'The Essential Miles Davis [Disc 1]',
  "'Round Midnight",
  'Miles Davis',
  357459,
  'Miles Davis'),
 (150,
  "Kill 'Em All",
  '(Anesthesia) Pulling Teeth',
  'Metallica',
  254955,
  'Cliff Burton'),
 (46, 'Supernatural', '(Da Le) Yaleo', 'Santana', 353488, 'Santana'),
 (241,
  'UB40 The Best Of - Volume Two [UK]',
  "(I Can't Help) Falling In Love With You",
  'UB40',
  207568,
  None),
 (242,
  'Diver Down',
  '(Oh) Pretty Woma

### Tools with states

The following isn an example of a more complex tool that requires initialisation

In [ ]:
class SQLiteTool(Tool):
   # tools name
   name = "album_track"

   description = """
    Perform SQL query on the album_track table. Returns the result as an array of tuples.
    The table name is album_track and the schema is as follows:
      AlbumId: integer
      Title: nvarchar(160)
      track_name: nvarchar(200)
      artist_name: nvarchar(120)
      duration: integer
      composer: nvarchar(220)
    The duration is in milliseconds
   """
   # Args:
   inputs = {
      "query": {
          "type": "string",
          "description": "A valid SQL query in SQLite dialect"
      }
   }

   # Returns:
   output_type = "string"

   def __init__(self, db_file):
      self.db_file = db_file
      self.setup()

   def setup(self):
      super().setup()
      self.conn = sqlite3.connect(self.db_file)

   def forward(self, query: str) -> str:
      try:
         cursor = self.conn.cursor()
         rows = cursor.execute(query)
         return rows.fetchall()
      except Exception as e:
         print(f'Query exception: {e}')